# Simple Data Collection Tool (Raw Image Capture)

This notebook captures raw camera snapshots from the **JetRacer CSI Camera** and saves them into a selected dataset folder for later labeling.

### 1. Initialize CSI Camera

In [ ]:
import os
import cv2
import glob
import time
import ipywidgets
from IPython.display import display
from jetcam.csi_camera import CSICamera
from jetcam.utils import bgr8_to_jpeg

# Initialize CSI Camera (224x224 at 65 FPS)
camera = CSICamera(width=224, height=224, capture_fps=65)
camera.running = True
print(f"CSI Camera started successfully: {camera.width}x{camera.height}")


### 2. Interactive Data Collection UI

* Select or type a target **dataset folder**.
* Click **Save Frame (Snapshot)** to save raw images. You can annotate them later.

In [ ]:
import os
import cv2
import glob
import time
import datetime
import ipywidgets
from IPython.display import display
from ipywidgets import Layout
from jetcam.utils import bgr8_to_jpeg

# Root directory for saving datasets
DATASET_ROOT = 'dataset'
os.makedirs(DATASET_ROOT, exist_ok=True)

# Widget UI Elements
widget_style = {'description_width': '140px'}

dataset_dropdown = ipywidgets.Dropdown(
    options=['dataset_A', 'dataset_B', 'dataset_C', 'custom'],
    value='dataset_A',
    description='Select Dataset:',
    style=widget_style,
    layout=Layout(width='380px')
)

custom_name_text = ipywidgets.Text(
    value='',
    placeholder='Type new folder name...',
    description='Custom Folder:',
    disabled=True,
    style=widget_style,
    layout=Layout(width='380px')
)

count_widget = ipywidgets.IntText(
    value=0,
    description='Total Images:',
    disabled=True,
    style=widget_style,
    layout=Layout(width='380px')
)

status_label = ipywidgets.Label(value='Ready to collect images.')

# Camera live stream & last saved snapshot preview
camera_widget = ipywidgets.Image(value=bgr8_to_jpeg(camera.value), format='jpeg', width=camera.width, height=camera.height)
snapshot_widget = ipywidgets.Image(format='jpeg', width=camera.width, height=camera.height)

# Action Buttons
save_button = ipywidgets.Button(description='Save Frame (Snapshot)', button_style='success', icon='camera', layout=Layout(width='200px', height='40px'))
refresh_count_btn = ipywidgets.Button(description='Refresh Count', button_style='info', icon='refresh', layout=Layout(width='160px', height='40px'))

def get_active_dataset_dir():
    if dataset_dropdown.value == 'custom':
        folder_name = custom_name_text.value.strip()
        if not folder_name:
            folder_name = 'dataset_custom'
    else:
        folder_name = dataset_dropdown.value
        
    target_dir = os.path.join(DATASET_ROOT, folder_name)
    os.makedirs(target_dir, exist_ok=True)
    return target_dir

def update_image_count(b=None):
    target_dir = get_active_dataset_dir()
    existing = glob.glob(os.path.join(target_dir, "*.jpg"))
    count_widget.value = len(existing)
    status_label.value = f"Active Directory: '{target_dir}' ({len(existing)} images)"

def on_dataset_change(change):
    if change['new'] == 'custom':
        custom_name_text.disabled = False
    else:
        custom_name_text.disabled = True
    update_image_count()

dataset_dropdown.observe(on_dataset_change, names='value')
custom_name_text.observe(lambda change: update_image_count(), names='value')

def save_snapshot(b):
    try:
        target_dir = get_active_dataset_dir()
        frame = camera.value.copy()
        timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:21]
        img_filename = f"img_{timestamp}.jpg"
        save_path = os.path.join(target_dir, img_filename)
        
        cv2.imwrite(save_path, frame)
        
        preview = frame.copy()
        cv2.putText(preview, "SAVED OK", (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
        cv2.circle(preview, (200, 20), 8, (0, 255, 0), -1)
        snapshot_widget.value = bgr8_to_jpeg(preview)
        
        update_image_count()
        status_label.value = f"✅ Saved: {img_filename} -> {target_dir}"
        
    except Exception as e:
        status_label.value = f"❌ Error saving frame: {e}"

save_button.on_click(save_snapshot)
refresh_count_btn.on_click(update_image_count)
update_image_count()

top_box = ipywidgets.HBox([
    ipywidgets.VBox([ipywidgets.HTML("<b>Live Camera Stream:</b>"), camera_widget]),
    ipywidgets.VBox([ipywidgets.HTML("<b>Last Saved Snapshot:</b>"), snapshot_widget])
], layout=Layout(justify_content='space-around', margin='0px 0px 15px 0px'))

controls_box = ipywidgets.VBox([
    dataset_dropdown,
    custom_name_text,
    count_widget,
    ipywidgets.HBox([save_button, refresh_count_btn], layout=Layout(margin='10px 0px 10px 0px')),
    status_label
], layout=Layout(align_items='center'))

display(ipywidgets.VBox([top_box, controls_box]))
